# Spark streaming

In [6]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.config("spark.sql.streaming.schemaInference", True).getOrCreate()

stream = spark.\
    readStream.\
    format("ws").\
    option("schema", "ticker").\
    load() # we need to pass `option("schema", "ticker")` to get correct channel subscribed

query = stream.select("side", "product_id", "last_size", "best_bid", "best_ask", "time").\
    writeStream.\
    format("console").\
    outputMode("append").\
    option("truncate", "false").\
    start()

query.awaitTermination(10) # Let's wait for 10 seconds.
query.stop() # Let's stop the query
stream.printSchema()
#spark.stop() # And stop the whole session

root
 |-- type: string (nullable = false)
 |-- trade_id: long (nullable = false)
 |-- sequence: long (nullable = false)
 |-- time: timestamp (nullable = false)
 |-- product_id: string (nullable = false)
 |-- price: double (nullable = false)
 |-- side: string (nullable = false)
 |-- last_size: double (nullable = false)
 |-- best_bid: double (nullable = false)
 |-- best_ask: double (nullable = false)



In [21]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.config("spark.sql.streaming.schemaInference", True).getOrCreate()

stream = spark.\
    readStream.\
    format("ws").\
    option("schema", "heartbeat").\
    load() # we need to pass `option("schema", "ticker")` to get correct channel subscribed

# query = stream.select("side", "product_id", "last_size", "best_bid", "best_ask", "time").\
#     writeStream.\
#     format("console").\
#     outputMode("append").\
#     option("truncate", "false").\
#     start()

# query.awaitTermination(10) # Let's wait for 10 seconds.
# query.stop() # Let's stop the query
stream.printSchema()
#spark.stop() # And stop the whole session

root
 |-- type: string (nullable = false)
 |-- sequence: long (nullable = false)
 |-- last_trade_id: long (nullable = false)
 |-- product_id: string (nullable = false)
 |-- time: timestamp (nullable = false)



Uruchamiająć `stream.start()` uruchamiamy w osobnym demonie websocket który streamuje wyniki. Jeżeli wystąpi jakiś błąd po stronie front-endu (np. błąd parsowania kolejnej linijki Pythona) fakt ten nie zostanie zgłoszony do sparka i socket pozostanie otwarty! Należy pamiętać, by zamykać stream za każdym razem używająć metody `stop()` (w powyższym przykładzie `query.stop()`). W przypadku utracenia referencji do zapytania, należy zastopować całą sesję również metodą `stop()` (w powyższym przykładzie `spark.stop()`) 

In [2]:
# panic button - press only if you messed up opening new websocket and lost reference to it

query.stop()
spark.stop()

## Zadanie 1

**Analiza strumienia danych CoinBase (2p)**. Napisz zapytanie, które wypisuje średnią wartość wybranego parametru (np. `price`) w przesuwnych oknach czasowych względem czasu transakcji (kolumna `time`), grupując po relacji wymiany (z jakiej waluty na jaką walutę - kolumna `product_id`).

In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.config("spark.sql.streaming.schemaInference", True).getOrCreate()

stream = spark.readStream \
    .format("ws") \
    .option("schema", "ticker") \
    .load()

query = stream.select("product_id", "price", "time") \
    .groupBy("product_id", F.window("time", "5 minutes", "2 minutes")) \
    .agg(F.avg("price").alias("average_price")) \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", "false") \
    .start()

query.awaitTermination(60)
query.stop()

```
Batch: 0
-------------------------------------------
+----------+------+---------+
|product_id|window|avg_price|
+----------+------+---------+
+----------+------+---------+

-------------------------------------------                                     
Batch: 1
-------------------------------------------
+----------+------------------------------------------+------------------+
|product_id|window                                    |avg_price         |
+----------+------------------------------------------+------------------+
|ETH-USD   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|3199.6196052631576|
|BTC-EUR   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|85141.651         |
|ETH-USD   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|3199.6196052631576|
|ETH-USD   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|3199.6196052631576|
|ETH-BTC   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|0.03527           |
|BTC-EUR   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|85141.651         |
|BTC-USD   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|90731.64135964912 |
|ETH-BTC   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|0.03527           |
|ETH-BTC   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|0.03527           |
|BTC-USD   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|90731.64135964912 |
|BTC-EUR   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|85141.651         |
|BTC-USD   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|90731.64135964912 |
+----------+------------------------------------------+------------------+

[I 2025-12-02 16:19:13.511 ServerApp] Saving file at /work/Spark_streaming.ipynb
[I 2025-12-02 16:19:19.863 ServerApp] Saving file at /work/lab4.ipynb+ 4) / 200]
-------------------------------------------                                     
Batch: 2
-------------------------------------------
+----------+------------------------------------------+------------------+
|product_id|window                                    |avg_price         |
+----------+------------------------------------------+------------------+
|ETH-USD   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|3200.1989847715736|
|BTC-EUR   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|85159.18416666666 |
|ETH-USD   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|3199.794421052631 |
|ETH-USD   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|3199.794421052631 |
|ETH-BTC   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|0.03527           |
|BTC-EUR   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|85166.00066666667 |
|BTC-USD   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|90745.77653658538 |
|ETH-BTC   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|0.0352675         |
|ETH-BTC   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|0.0352675         |
|BTC-USD   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|90745.77653658538 |
|BTC-EUR   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|85159.18416666666 |
|BTC-USD   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|90763.33          |
+----------+------------------------------------------+------------------+

-------------------------------------------                                     
Batch: 3
-------------------------------------------
+----------+------------------------------------------+-------------------+
|product_id|window                                    |avg_price          |
+----------+------------------------------------------+-------------------+
|ETH-USD   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|3200.1989847715736 |
|BTC-EUR   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|85035.67445652172  |
|ETH-USD   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|3195.0106576728494 |
|ETH-USD   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|3195.0106576728494 |
|ETH-BTC   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|0.03527            |
|BTC-EUR   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|85166.00066666667  |
|BTC-USD   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|90659.98448589628  |
|ETH-BTC   |{2025-12-02 16:16:00, 2025-12-02 16:21:00}|0.03526269230769231|
|ETH-BTC   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|0.03526269230769231|
|BTC-USD   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|90659.98448589628  |
|BTC-EUR   |{2025-12-02 16:18:00, 2025-12-02 16:23:00}|85035.67445652172  |
|BTC-USD   |{2025-12-02 16:14:00, 2025-12-02 16:19:00}|90763.33           |
+----------+------------------------------------------+-------------------+
```

# Zadanie 2

**Watermarking i dane opóźnione (2p).** 
Zmodyfikuj zapytanie z zadania 1 tak, aby zademonstować mechanizm znaków wodnych (watermarks) i obsługi danych opóźnionych. W konsoli powinno być widać, że aktualizują się odpowiednie wiersze tabeli wynikowej (tryb update), w szczególności aktualizacja wcześniejszych okien czasowych po przybyciu danych opóźnionych. **Do rozwiązania tego zadania proszę dołączyć przykładowy output i jego opis wyjaśniający na konkretnym przykładzie działanie znaku wodnego i danych opóźnionych**. 

Do ćwiczenia można wykorzystać skrypt w katalogu `/mock` napisany w [Scala-cli](https://scala-cli.virtuslab.org), który posłuży jako kontrolowane źródło danych CoinBase przez Websocket. 

Skrypt można uruchomić wykorzystując Docker:

```
make image
make run
```

Spowoduje to utworzenie websocketowego serwera pod adresem `ws://mock:8025`

Po uruchomieniu serwera należy wykonać poniższą komórkę, w której zapytanie czyta dane z utworzonego websocketa. Skrypt wysyła przykładowe wiadomości w formacie CoinBase co 10 sekund:

- W pierwszej serii wysyłane wiadomości o znacznikach czasowych 0s, 14s, 7s  
- W drugiej serii wysyłane są wiadomości o znacznikach czasowych 15s, 8s, 21s  
- W trzeciej serii wysyłane są wiadomości o znacznikach czasowych 4s, 17s  

Dla tych danych można ustawić okno czasowe na interwał 10 sekund. Skrypt można też zmodyfikować, tak aby wysyłał inne dane. 

In [4]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .config("spark.sql.streaming.schemaInference", True) \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", True)\
    .getOrCreate()

stream = spark.readStream.format("ws") .option("schema", "ticker").option("url", "ws://mock:8025") .load()

query = stream.select("product_id", "price", "time") \
    .withWatermark("time", "10 seconds") \
    .groupBy( "product_id", F.window("time", "10 seconds"))\
    .agg(F.avg("price").alias("avg_price"))\
    .writeStream.format("console")\
    .outputMode("update")\
    .option("truncate", "false") \
    .start()

query.awaitTermination(120)
query.stop()


```
-------------------------------------------                                     
Batch: 0
-------------------------------------------
+----------+------+---------+
|product_id|window|avg_price|
+----------+------+---------+
+----------+------+---------+

-------------------------------------------                                     
Batch: 1
-------------------------------------------
+----------+------------------------------------------+-----------------+------+
|product_id|window                                    |avg_price        |tmstp |
+----------+------------------------------------------+-----------------+------+
|ETH-USD   |{2021-11-01 00:00:10, 2021-11-01 00:00:20}|663.6292543013998|14s   |
|ETH-USD   |{2021-11-01 00:00:00, 2021-11-01 00:00:10}|701.122284534661 |0s, 7s|
+----------+------------------------------------------+-----------------+------+

-------------------------------------------                                     
Batch: 2
-------------------------------------------
+----------+------------------------------------------+------------------+-----+
|product_id|window                                    |avg_price         |tmstp|
+----------+------------------------------------------+------------------+-----+
|ETH-USD   |{2021-11-01 00:00:10, 2021-11-01 00:00:20}|341.49609080688754|15s  |
|ETH-USD   |{2021-11-01 00:00:00, 2021-11-01 00:00:10}|547.7594552669779 |8s   |
|ETH-USD   |{2021-11-01 00:00:20, 2021-11-01 00:00:30}|856.8793108133178 |0s   |
+----------+------------------------------------------+------------------+-----+

-------------------------------------------                                     
Batch: 3
-------------------------------------------
+----------+------+---------+
|product_id|window|avg_price|
+----------+------+---------+
+----------+------+---------+

-------------------------------------------                                     
Batch: 4
-------------------------------------------
+----------+------------------------------------------+------------------+----+
|product_id|window                                    |avg_price         |tmsp|
+----------+------------------------------------------+------------------+----+
|ETH-USD   |{2021-11-01 00:00:10, 2021-11-01 00:00:20}|376.28034933449766|17s |
+----------+------------------------------------------+------------------+----+
+-----------------+----------+------------------+
```

# Zadanie 3

**Łączenie strumieni (1p)**. Korzystając z łączenia strumieni połącz dane z kanału `ticker` (transakcje kupna `side="buy"`) razem z danymi o transakcjach napływających co sekundę `heartbeat` korzystając z `trade_id` i odpowiedniego id w kanale heartbeat. Wypisz połączony strumień danych.

Na moment tworzenia zadania 15.11.2023 kanał `heartbeat` zwraca błędne dane o dacie (np. `1970-01-04 13:53:57.645339`). Połączenie z kanałem `ticker` pozwala uzyskać poprawne informacje. Cóż za wspaniałe zastosowanie joina!

In [5]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.config("spark.sql.streaming.schemaInference", True) \
        .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", True).getOrCreate()

stream1 = spark.readStream.format("ws").option("schema", "ticker").load()

stream2 = spark.readStream.format("ws").option("schema", "heartbeat").load()

buy = stream1.select("side", "product_id", "price", "time") \
    .filter(stream1.side == "buy") \
    .withColumnRenamed("side", "buy_side") \
    .withColumnRenamed("product_id", "buy_product_id") \
    .withColumnRenamed("price", "buy_price") \
    .withColumnRenamed("time", "buy_time")

beat = stream2.select("last_trade_id", "product_id", "type", "sequence")


joined = buy.join(beat, F.expr(
            """
            product_id = buy_product_id
            """
            ))

query = joined.writeStream .format("console").outputMode("append").option("truncate", "false").start()
query.awaitTermination(60)
query.stop()

```
Batch: 0
-------------------------------------------
+--------+--------------+---------+--------+-------------+----------+----+--------+
|buy_side|buy_product_id|buy_price|buy_time|last_trade_id|product_id|type|sequence|
+--------+--------------+---------+--------+-------------+----------+----+--------+
+--------+--------------+---------+--------+-------------+----------+----+--------+

-------------------------------------------                                     
Batch: 1
-------------------------------------------
+--------+--------------+---------+-----------------------+-------------+----------+---------+----------+
|buy_side|buy_product_id|buy_price|buy_time               |last_trade_id|product_id|type     |sequence  |
+--------+--------------+---------+-----------------------+-------------+----------+---------+----------+
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480778     |ETH-BTC   |heartbeat|7570882877|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480778     |ETH-BTC   |heartbeat|7570882877|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570883224|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570883224|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570883425|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570883425|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570883681|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570883681|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570883777|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570883777|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570883995|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570883995|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570884183|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570884183|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570884377|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570884377|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570884472|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570884472|
|buy     |ETH-BTC       |0.03513  |2025-12-02 16:32:23.083|34480781     |ETH-BTC   |heartbeat|7570884666|
|buy     |ETH-BTC       |0.03515  |2025-12-02 16:32:17.798|34480781     |ETH-BTC   |heartbeat|7570884666|
+--------+--------------+---------+-----------------------+-------------+----------+---------+----------+
only showing top 20 rows
```